# STREAMING

**Streaming in LlamaIndex means the LLM response is generated and displayed token-by-token instead of waiting for the complete answer.**

<div align="center">
    <img src="images/streaming.png" width="300" height="700">
</div>

<h1 style="color: #16A34A;">1. STREAMING WITH QUERY ENGINE</h1>

**streaming=True ---> This tells the query engine to request a streaming response from the LLM.**<br>
**print_response_stream() ---> This actually prints the streamed chunks to the console.**<br>

In [2]:
import logging
logging.getLogger("opentelemetry").setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry.exporter.otlp.proto.http.trace_exporter").setLevel(logging.CRITICAL)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [3]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
llm = OpenAI(model="gpt-4o-mini", temperature=0, streaming=True)
documents = SimpleDirectoryReader(input_files=["data/HopeAI_Teaching.pdf"]).load_data()

2026-09-29 13:00:56,707 - INFO - NumExpr defaulting to 12 threads.


In [4]:
# Create index
index = VectorStoreIndex.from_documents(documents)

# Create query engine with streaming
query_engine = index.as_query_engine(streaming=True)

# Ask question
response = query_engine.query("What is artificial intelligence?")

# Stream the response
response.print_response_stream()

C:\Users\User\anaconda3\envs\ai\Lib\site-packages\pydantic\_internal\_generate_schema.py:937: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `BasePromptTemplate` to V2.
  warnings.warn(


Artificial intelligence refers to the ability of computer systems to perform tasks that typically require human intelligence, such as learning, reasoning, recognizing patterns, understanding information, making predictions, and supporting decisions. It involves using algorithms and data to execute tasks that appear intelligent within a specific context.

<h1 style="color: #16A34A;">2. STREAMING WITH CHAT ENGINE</h1>

In [5]:
index = VectorStoreIndex.from_documents(documents)

chat_engine = index.as_chat_engine(streaming=True)

response = chat_engine.stream_chat("Explain RAG in simple terms")

for chunk in response.response_gen:
    print(chunk, end="", flush=True)

2026-09-29 13:01:12,721 - INFO - Condensed question: Explain RAG in simple terms


Retrieval-augmented generation (RAG) is a method that combines information retrieval with language generation. When you give an input to the system, it searches for relevant information from external sources like documents or knowledge bases. This retrieved information is then used by a language model to help generate a response. 

In simpler terms, RAG is like having a smart assistant that looks up helpful information for you when you ask a question, and then uses that information to give you a well-informed answer. It's a way to make sure the responses generated by AI are based on real, relevant facts rather than just what the AI has been trained on.

<h1 style="color: #6A11CB;">SYNCHRONOUS & ASYNCHRONOUS STREAMING</h1>

**Synchronous streaming: receive and process streamed data sequentially while the current operation waits.**<br>
**Asynchronous streaming: receive and process streamed data asynchronously, allowing other async work to run while waiting.**

<div align="center">
    <img src="images/stream2.png" width="500" height="700">
</div>

<h1 style="color: #16A34A;">3. SYNCHRONOUS STREAMING</h1>

In [6]:
# Create index
index = VectorStoreIndex.from_documents(documents)

# Create query engine with streaming
query_engine = index.as_query_engine(streaming=True)

# Ask question
response = query_engine.query("What is artificial intelligence?")

# Print response progressively
response.print_response_stream()

Artificial intelligence refers to the ability of computer systems to perform tasks that typically require human intelligence, such as learning, reasoning, recognizing patterns, understanding information, making predictions, and supporting decisions. It involves using algorithms and data to carry out tasks that appear intelligent within a specific context.

<h1 style="color: #16A34A;">4. ASYNCHRONOUS STREAMING</h1>

<div align="center">
    <img src="images/async.png" width="500" height="700">
</div>

In [7]:
import asyncio

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(streaming=True)

async def main():
    response = await query_engine.aquery("What is artificial intelligence?")
    
    async for chunk in response.async_response_gen():
        print(chunk, end="", flush=True)

await main()

Artificial intelligence refers to the ability of computer systems to perform tasks that typically require human intelligence, such as learning, reasoning, recognizing patterns, understanding information, making predictions, and supporting decisions. It involves using algorithms and data to execute tasks that appear intelligent within a specific context.